# Week 2 — Fast Pipelines. Отчёт

---

## Task 1: DIY Loss Scaling (2 балла)

Реализованы два режима loss scaling для AMP-тренировки модели сегментации (U-Net на датасете Carvana). Весь код находится в `task1/train.py`.

Тренировка выполняется в AMP-режиме (`torch.amp.autocast` с `dtype=torch.float16`). Forward pass и вычисление loss происходят в FP16, а backward и обновление весов — в FP32. Loss scaling предотвращает обнуление малых градиентов при FP16-вычислениях.

### Static Loss Scaling (1 балл)

Класс `StaticScaler` — фиксированный множитель `scale_factor = 2^10` на протяжении всей тренировки.

Алгоритм:
1. `scale(loss)` — умножает loss на `scale_factor` перед `.backward()`, чтобы FP16-градиенты не обнулялись из-за underflow.
2. `step(optimizer)` — перед шагом оптимизатора делит все градиенты на `scale_factor` (unscale), возвращая их к истинным значениям. После этого вызывает `optimizer.step()`.
3. `update()` — no-op, множитель не меняется.

### Dynamic Loss Scaling (1 балл)

Класс `DynamicScaler` — адаптивный множитель, начинающий с `2^16` и подстраивающийся в процессе тренировки.

Алгоритм:
1. `scale(loss)` — аналогично static, умножает loss на текущий `scale_factor`.
2. `step(optimizer)` — перед unscale проверяет все градиенты на `inf`/`nan`. Если найдены — обнуляет градиенты и пропускает шаг оптимизатора (выставляет флаг `_found_inf`). Если градиенты корректны — unscale и `optimizer.step()`.
3. `update()` — если на текущем шаге был overflow (`_found_inf`), уменьшает `scale_factor` в 2 раза (`backoff_factor=0.5`). Иначе увеличивает счётчик успешных шагов; после `growth_interval=2000` подряд успешных шагов увеличивает `scale_factor` в 2 раза (`growth_factor=2.0`).

### Training loop

```
optimizer.zero_grad()
with torch.amp.autocast("cuda", dtype=torch.float16):
    outputs = model(images)
    loss = criterion(outputs, labels)
scaled_loss = scaler.scale(loss)
scaled_loss.backward()
scaler.step(optimizer)
scaler.update()
```

### Как запустить

```bash
cd task1
python train.py static   # StaticScaler
python train.py dynamic  # DynamicScaler
python train.py none     # AMP без скейлера (для сравнения)
```

Тренировка 5 эпох, ожидаемая accuracy 0.985+ с обоими скейлерами.

### Запуски

In [ ]:
import subprocess, sys

# Запуск 1: AMP без скейлера (baseline для сравнения)
result = subprocess.run(
    [sys.executable, "train.py", "none"],
    cwd="task1",
    capture_output=False,
    text=True,
)
print("Return code:", result.returncode)

In [ ]:
import subprocess, sys

# Запуск 2: StaticScaler
result = subprocess.run(
    [sys.executable, "train.py", "static"],
    cwd="task1",
    capture_output=False,
    text=True,
)
print("Return code:", result.returncode)

In [ ]:
import subprocess, sys

# Запуск 3: DynamicScaler
result = subprocess.run(
    [sys.executable, "train.py", "dynamic"],
    cwd="task1",
    capture_output=False,
    text=True,
)
print("Return code:", result.returncode)

## Task 2: Efficient Batching for Language Modeling (5 баллов)

Реализованы 4 подхода к батчингу для языковой модели на датасете WikiText-103. Все реализации находятся в `task2/dataset.py` и `task2/run_epoch.py`.

Токенизация: `AutoTokenizer.from_pretrained("bert-base-uncased")`, метод `.tokenize()`. Последовательности длиннее 640 токенов отбрасываются. Паддинг реализован вручную (token id = 0).

### Подзадача 1: BRAIN (0.5 балла)

Класс `BrainDataset` — каждая последовательность паддится до фиксированной длины `max_length=640` прямо в `__getitem__`. Возвращает пару `(input_ids, targets)` со сдвигом на 1 позицию.

### Подзадача 2: BIG BRAIN (0.5 балла)

Класс `BigBrainDataset` — возвращает сырые тензоры токенов. Паддинг выполняется в `collate_fn`: все последовательности в батче паддятся до максимальной длины в текущем батче (`max(lengths)`), а не до глобального `max_length`.

### Подзадача 3: ULTRA BIG BRAIN (1.5 балла)

Класс `UltraBigBrainDataset` — хранит последовательности и их длины. Класс `UltraBigBrainBatchSampler` — bucket sampler:
- `__init__` (O(n)): строит хеш-таблицу `length → [indices]`, затем группирует уникальные длины в бакеты, где разница между максимальной и минимальной длиной ≤ `k`.
- `__iter__` (O(batch_size) относительно размера датасета): перемешивает индексы внутри каждого бакета и выдаёт батчи. Батчи между бакетами тоже перемешиваются.

При `k=640` все последовательности попадают в один бакет — результаты совпадают с подзадачей 2.

### Подзадача 4: ULTRA DUPER BIG BRAIN (2.5 балла)

Класс `UltraDuperBigBrainDataset` — sequence packing с тремя режимами:

**Basic packing (0.5 балла):** жадная конкатенация — последовательности складываются подряд, пока помещаются в `max_length`. Результат паддится до `max_length`. Маска внимания (segment IDs) предотвращает cross-contamination: каждый токен знает, к какой исходной последовательности он принадлежит, паддинг помечается как -1.

**FFD packing (1 балл):** First-Fit Decreasing — последовательности сортируются по убыванию длины, каждая помещается в первый бин, где хватает места. Если не помещается — создаётся новый бин. Последовательности длиннее `max_length` пропускаются. Сложность O(N * M).

**OBFD packing (1 балл):** Optimized Best-Fit Decreasing — вместо линейного поиска бина используется **дерево отрезков** (segment tree) по оставшимся ёмкостям бинов. Для каждой последовательности за O(log L) находится бин с минимальной оставшейся ёмкостью, достаточной для размещения. Общая сложность O(N log L), где L = max_length.

### Модель

Класс `GPT2Model` в `run_epoch.py` — упрощённая GPT-2: `nn.Embedding` + `PositionalEncoding` (из `transformer.py`) + один `nn.TransformerDecoderLayer` (d_model=1024, 8 голов). Для packed-последовательностей строится блочно-диагональная маска внимания, комбинированная с каузальной маской.

### Бенчмарк

Функция `run_epoch()` запускает один проход (только forward, без backward) по всему датасету. Первые 5 батчей — warmup. Замеры с `torch.cuda.synchronize()` перед стартом и после завершения каждого батча. Собираются min, max, mean, median времени обработки батча.

**Как запустить:**
```bash
cd task2
python run_epoch.py path/to/wiki.train.raw
```

---

## Task 3: Профилирование ViT и поиск неэффективностей (4 балла)

### 3.1 Кастомный профайлер (1.5 балла)

Реализован класс `Profile` (`task3/profiler.py`) — контекстный менеджер для послойного профилирования PyTorch-моделей.

**Основные возможности:**
- Регистрация forward/backward хуков на все модули модели для замера времени выполнения каждого слоя.
- На GPU используются `torch.cuda.Event(enable_timing=True)` для точного замера времени с учётом асинхронного выполнения CUDA-ядер. На CPU — `time.perf_counter()`.
- Поддержка **расписания профилирования** (фазы `wait`, `warmup`, `active`), аналогично `torch.profiler.schedule`. Во время фаз `wait` и `warmup` события не записываются.
- Метод `step()` для перехода между фазами расписания.
- Метод `to_perfetto()` экспортирует собранные события в формат [Trace Event Format](https://docs.google.com/document/d/1CvAClvFfyA5R-PhYUmn5OOQtYMH4h6I0nSsKchNAySU/preview), совместимый с [Perfetto UI](https://ui.perfetto.dev/).
- Метод `summary()` выводит агрегированную статистику по слоям.

### 3.2 Измерение производительности (0.5 балла)

Профилирование ViT выполняется функцией `run_epoch_with_profiler()` в `task3/run_epoch.py`.

**Как запустить:**
```bash
cd task3
python run_epoch.py
```

После запуска генерируется файл `trace.json`, который можно загрузить в [Perfetto UI](https://ui.perfetto.dev/) для визуализации.

**Сравнение с PyTorch profiler:**
- Кастомный профайлер измеряет время на уровне слоёв (layer-level), тогда как `torch.profiler` работает на уровне CUDA-ядер (kernel-level).
- Время отдельных слоёв в кастомном профайлере включает overhead от хуков и `torch.cuda.synchronize()`, поэтому абсолютные значения могут быть немного выше.
- Относительные пропорции времени между слоями при этом сохраняются и совпадают с данными PyTorch profiler.

### 3.3 Найденные и исправленные неэффективности (2 балла)

В коде было найдено и исправлено **6 намеренных неэффективностей**:

| № | Файл | Строка | Проблема | Исправление |
|---|------|--------|----------|-------------|
| 1 | `vit.py` | 95 | `dim=255` — размерность не кратна степени двойки. GPU tensor cores оптимизированы для размерностей, кратных 8/16/32. Нечётная размерность 255 приводит к неоптимальному использованию памяти и вычислительных ресурсов. | Изменено на `dim=256` |
| 2 | `vit.py` | 17 | `hidden_dim=255` — та же проблема для скрытого слоя FeedForward. | Изменено на `hidden_dim=256` |
| 3 | `run_epoch.py` | 76–77 | В цикле валидации метрики нормализуются на `len(train_loader)` вместо `len(val_loader)` — **баг**, приводящий к некорректным значениям val accuracy/loss. | Исправлено на `len(val_loader)` |
| 4 | `run_epoch.py` | 69–84 | Валидация выполняется **без `torch.no_grad()`** — PyTorch строит вычислительный граф и хранит промежуточные активации для обратного прохода, который никогда не выполняется. Это тратит GPU-память и замедляет inference. | Обёрнуто в `torch.no_grad()` |
| 5 | `dataset.py` | 54–56 | В train-трансформах выполняется `Resize(320) → CenterCrop(224) → RandomResizedCrop(224)`. `CenterCrop` перед `RandomResizedCrop` **избыточен**: он обрезает изображение до 224×224, после чего `RandomResizedCrop` работает с уже обрезанным изображением, теряя информацию с краёв. Это и лишние вычисления, и ухудшение аугментации. | Удалён `CenterCrop` из цепочки |
| 6 | `run_epoch.py` | 47–48 | `DataLoader` создаётся с `num_workers=0` (загрузка данных в основном потоке) и без `pin_memory`. Это создаёт узкое место: GPU простаивает, пока CPU загружает и трансформирует данные. | Добавлены `num_workers=4`, `pin_memory=True`, а также `non_blocking=True` при `.to(device)` |

#### Анализ по времени и памяти

**Время:**
- Исправления 1–2 (`dim=256`): ускорение матричных умножений за счёт выравнивания на tensor cores. Эффект наиболее заметен на GPU с поддержкой Tensor Cores (Volta+).
- Исправление 4 (`no_grad`): сокращение времени валидации, так как PyTorch не строит вычислительный граф.
- Исправление 5 (удаление `CenterCrop`): небольшое ускорение предобработки за счёт удаления одной операции трансформации.
- Исправление 6 (`num_workers`): значительное ускорение за счёт параллельной загрузки данных — GPU не простаивает в ожидании батча.

**Память:**
- Исправление 4 (`no_grad`): наибольший эффект — PyTorch не хранит промежуточные активации для backward pass во время валидации, что существенно снижает пиковое потребление GPU-памяти.
- Исправления 1–2: незначительное изменение (256 vs 255 — разница в 1 элемент на вектор), но padding до кратного размера может быть эффективнее по аллокации.